# Restaurant Annual Turnover Prediction

This notebook follows the supplied sample notebook's **load → inspect → prepare → model → predict → save** flow. It adds a reliable validation score and uses all available predictors.

**Goal:** predict `Annual Turnover`. **Competition metric:** RMSE (lower is better).

## 1. Import libraries and locate the project files

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')

# This works when the notebook is launched from either Hackathon/ or Hackathon/notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'Train_dataset.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_PATH = PROJECT_ROOT / 'Train_dataset.csv'
TEST_PATH = PROJECT_ROOT / 'Test_dataset.csv'
SUBMISSION_DIR = PROJECT_ROOT / 'submissions'
TARGET = 'Annual Turnover'
ID_COLUMN = 'Registration Number'
RANDOM_STATE = 42

print('Project root:', PROJECT_ROOT.resolve())

## 2. Load and understand the data

In [ ]:
train = pd.read_csv(TRAIN_PATH, na_values=['NA'])
test = pd.read_csv(TEST_PATH, na_values=['NA'])

print('Training shape:', train.shape)
print('Test shape:    ', test.shape)
train.head()

In [ ]:
# The target should appear only in training data.
print('Target in training data:', TARGET in train.columns)
print('Target in test data:    ', TARGET in test.columns)

column_summary = pd.DataFrame({
    'dtype': train.dtypes.astype(str),
    'missing_count': train.isna().sum(),
    'missing_percent': (train.isna().mean() * 100).round(1),
    'unique_values': train.nunique(dropna=True),
}).sort_values('missing_percent', ascending=False)
column_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(train[TARGET], kde=True, ax=axes[0])
axes[0].set_title('Annual Turnover Distribution')
sns.boxplot(x=train[TARGET], ax=axes[1])
axes[1].set_title('Annual Turnover and Possible Outliers')
plt.tight_layout();

## 3. Prepare features

We do not use the registration number as a predictor because it is an identifier. We turn the opening date into useful numeric features. The supplied test file also lacks `Endorsed By`, so we explicitly align it to the training columns.

In [ ]:
def add_date_features(frame):
    result = frame.copy()
    opened = pd.to_datetime(
        result.pop('Opening Day of Restaurant'),
        dayfirst=True,
        errors='coerce',
    )
    result['Opening Year'] = opened.dt.year
    result['Opening Month'] = opened.dt.month
    result['Restaurant Age'] = 2026 - opened.dt.year
    return result

y = train[TARGET].astype(float)
X = add_date_features(train.drop(columns=[TARGET, ID_COLUMN]))
X_test = add_date_features(test.drop(columns=[ID_COLUMN]))
X_test = X_test.reindex(columns=X.columns)

print('Modeling features:', X.shape[1])
print('Columns missing from original test file:', sorted(set(X.columns) - set(add_date_features(test.drop(columns=[ID_COLUMN])).columns)))

## 4. Build the preprocessing and model pipeline

A pipeline prevents data leakage: missing-value rules and category encoding are learned separately inside each validation fold.

In [ ]:
numeric_columns = X.select_dtypes(include='number').columns.tolist()
categorical_columns = X.select_dtypes(exclude='number').columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', min_frequency=5, sparse_output=False)),
])

preprocessing = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_columns),
    ('categorical', categorical_pipeline, categorical_columns),
])

model = ExtraTreesRegressor(
    n_estimators=400,
    min_samples_leaf=2,
    max_features=0.8,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('model', model),
])

print(f'{len(numeric_columns)} numeric and {len(categorical_columns)} categorical features')

## 5. Evaluate with cross-validation

Five-fold cross-validation trains on four parts and validates on the remaining part, repeating until every row has been held out once. This is much more trustworthy than evaluating on the same rows used for training.

In [ ]:
folds = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Keep this list small and intentional. Each run uses the same folds.
experiment_configs = [
    {'experiment_id': 'E00', 'min_samples_leaf': 2, 'max_features': 0.8, 'max_depth': None},
    {'experiment_id': 'E01', 'min_samples_leaf': 1, 'max_features': 0.8, 'max_depth': None},
    {'experiment_id': 'E02', 'min_samples_leaf': 4, 'max_features': 0.8, 'max_depth': None},
    {'experiment_id': 'E03', 'min_samples_leaf': 8, 'max_features': 0.8, 'max_depth': None},
    {'experiment_id': 'E04', 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': None},
    {'experiment_id': 'E05', 'min_samples_leaf': 2, 'max_features': 0.7, 'max_depth': None},
    {'experiment_id': 'E06', 'min_samples_leaf': 2, 'max_features': 1.0, 'max_depth': None},
    {'experiment_id': 'E07', 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None},
    {'experiment_id': 'E08', 'min_samples_leaf': 2, 'max_features': 0.8, 'max_depth': 20},
]

experiment_rows = []
for config in experiment_configs:
    candidate = clone(pipeline).set_params(
        model__min_samples_leaf=config['min_samples_leaf'],
        model__max_features=config['max_features'],
        model__max_depth=config['max_depth'],
    )
    negative_mse = cross_val_score(
        candidate, X, y, cv=folds, scoring='neg_mean_squared_error', n_jobs=1
    )
    fold_rmse = np.sqrt(-negative_mse)
    experiment_rows.append({
        **config,
        'mean_rmse': fold_rmse.mean(),
        'std_rmse': fold_rmse.std(),
    })
    print(f"{config['experiment_id']} complete: {fold_rmse.mean():,.0f} RMSE")

experiment_results = (
    pd.DataFrame(experiment_rows)
    .sort_values('mean_rmse')
    .reset_index(drop=True)
)
display(experiment_results.style.format({
    'mean_rmse': '{:,.0f}',
    'std_rmse': '{:,.0f}',
}))

best = experiment_results.iloc[0]
best_experiment_id = best['experiment_id']
best_mean_rmse = best['mean_rmse']
best_pipeline = clone(pipeline).set_params(
    model__min_samples_leaf=int(best['min_samples_leaf']),
    model__max_features=best['max_features'],
    model__max_depth=None if pd.isna(best['max_depth']) else int(best['max_depth']),
)
print(f'Best experiment: {best_experiment_id} with mean RMSE {best_mean_rmse:,.0f}')

### Experiment log — Extra Trees tuning

All runs used 5-fold shuffled cross-validation (`random_state=42`) and 400 trees. Results recorded from the same run on November 2, 2026.

- **E00 — Baseline:** leaf=2, features=0.8, unlimited depth. Mean RMSE **20,218,712**; std **2,624,634**. Observation: trustworthy starting point, but several alternatives improved it.
- **E01 — Smaller leaf:** leaf=1, features=0.8. Mean RMSE **20,715,176**; std **2,180,175**. Observation: lowest variation, but substantially worse mean; likely fitting noise. Reject.
- **E02 — Larger leaf:** leaf=4, features=0.8. Mean RMSE **20,202,836**; std **2,546,276**. Observation: tiny improvement over baseline; insufficient by itself.
- **E03 — Smoother trees:** leaf=8, features=0.8. Mean RMSE **20,065,181**; std **2,687,772**. Observation: useful improvement, suggesting the baseline slightly overfits.
- **E04 — Fewer sampled features:** leaf=2, features=0.5. Mean RMSE **20,052,211**; std **2,697,152**. Observation: strong runner-up; extra feature randomness improves generalization.
- **E05 — Moderate feature sampling:** leaf=2, features=0.7. Mean RMSE **20,134,932**; std **2,664,975**. Observation: better than baseline, but worse than sampling 50%.
- **E06 — All features:** leaf=2, features=1.0. Mean RMSE **20,505,073**; std **2,544,403**. Observation: worse; considering every feature reduces helpful tree diversity. Reject.
- **E07 — Square-root feature sampling:** leaf=2, features=`sqrt`. Mean RMSE **20,031,412**; std **2,673,136**. Observation: best mean in this batch, improving baseline by about **187,300 (0.93%)**. Select for this submission.
- **E08 — Depth limit:** leaf=2, features=0.8, depth=20. Mean RMSE **20,257,594**; std **2,613,569**. Observation: slightly worse than unlimited depth. Reject.

The top results are close relative to fold-to-fold variation, so E07 is a provisional winner rather than proof of a large breakthrough. The next useful test is combining `min_samples_leaf=8` with `max_features='sqrt'` or `0.5`.

## 6. Train on all rows and create the submission

After choosing a model using cross-validation, fit it on all labelled data and predict the unseen test data.

In [ ]:
best_pipeline.fit(X, y)
predictions = np.clip(best_pipeline.predict(X_test), a_min=0, a_max=None)

submission = pd.DataFrame({
    ID_COLUMN: test[ID_COLUMN],
    TARGET: predictions,
})
submission.head()

In [ ]:
# Validate before saving. The competition requires 500 rows and exactly two columns.
assert submission.shape == (500, 2)
assert submission.columns.tolist() == [ID_COLUMN, TARGET]
assert not submission.isna().any().any()
assert (submission[TARGET] >= 0).all()

SUBMISSION_DIR.mkdir(exist_ok=True)
submission_filename = f'submission_{best_experiment_id}_rmse_{best_mean_rmse:.0f}.csv'
submission_path = SUBMISSION_DIR / submission_filename
submission.to_csv(submission_path, index=False)
print('Saved:', submission_path.resolve())
print('Submission shape:', submission.shape)

## 7. Your next experiments

Change **one thing at a time** and record the mean RMSE:

1. Compare this model with `RandomForestRegressor` or gradient boosting.
2. Try predicting `np.log1p(Annual Turnover)` and convert predictions back with `np.expm1`.
3. Explore restaurant age, cuisine counts, popularity scores, and ratings.
4. Check whether extreme turnover values dominate RMSE.
5. Keep the same cross-validation folds so comparisons remain fair.

Do not use the test set to choose a model; it has no target and represents the final unseen data.